# Binary Classification Framework: One vs All Approach

This notebook implements binary classifiers using a one-vs-all strategy for multiclass fault detection in the 3W dataset. It combines:

- **Data structure** from the supervised learning approach with proper train/test separation and cross-validation folds
- **Architecture approach** from the OTC notebook with LSTM-based models for time series classification
- **Binary classification strategy** where each classifier distinguishes one specific fault class from all others

## Objectives

1. Load processed windowed data with cross-validation structure
2. Train binary classifiers (one vs all) for each fault class
3. Evaluate performance across different folds and classes
4. Compare accuracy and effectiveness of each binary classifier

## Key Features

- **Proper data separation**: Uses train/test splits and cross-validation folds
- **Windowed sequences**: Time series data prepared for LSTM models
- **One-vs-all strategy**: Individual binary classifiers for each fault type
- **Comprehensive evaluation**: Performance metrics across folds and classes

# Section 1: Load Processed Data

Load the processed windowed data from the data treatment pipeline including cross-validation folds and windowed sequences.

In [ ]:
# ============================================================
# LOAD 3W DATASET - Using Supervised Learning Structure
# ============================================================

import sys
import numpy as np
import pandas as pd
import os
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.append("../../src")
from src import config
from src.supervised_classification import (
    load_3w_data,
    validate_configuration,
    print_class_distribution_analysis,
)

print("Loading 3W Dataset for Binary Classification")
print("=" * 50)

# Load dataset using utility function from supervised learning approach
(train_dfs, train_classes, train_fold_info, test_dfs, test_classes, test_fold_info) = (
    load_3w_data(config, verbose=True)
)

print(f"✅ Data loaded successfully:")
print(f"   • Training windows: {len(train_dfs)}")
print(f"   • Test windows: {len(test_dfs)}")
print(f"   • Training classes distribution: {dict(Counter(train_classes))}")
print(f"   • Test classes distribution: {dict(Counter(test_classes))}")

# Store original data for reference
original_train_dfs = train_dfs.copy()
original_train_classes = train_classes.copy()
original_test_dfs = test_dfs.copy()
original_test_classes = test_classes.copy()

In [ ]:
# ============================================================
# CONFIGURE CLASSIFICATION FOR BINARY APPROACH
# ============================================================

from src.supervised_classification import (
    validate_configuration,
    print_class_distribution_analysis,
)

# Configuration for binary classification
# Exclude class 0 (normal operation) from all training and testing
all_available_classes = sorted(set(train_classes + test_classes))
selected_classes = [c for c in all_available_classes if c != 0]  # Exclude normal operation (class 0)

print(f"Filtering out class 0 from training and testing data...")

# Filter out class 0 from training data
train_indices_no_class0 = [i for i, cls in enumerate(train_classes) if cls != 0]
train_dfs = [train_dfs[i] for i in train_indices_no_class0]
train_classes = [train_classes[i] for i in train_indices_no_class0]
if train_fold_info:
    train_fold_info = [train_fold_info[i] for i in train_indices_no_class0]

# Filter out class 0 from test data
test_indices_no_class0 = [i for i, cls in enumerate(test_classes) if cls != 0]
test_dfs = [test_dfs[i] for i in test_indices_no_class0]
test_classes = [test_classes[i] for i in test_indices_no_class0]
if test_fold_info:
    test_fold_info = [test_fold_info[i] for i in test_indices_no_class0]

print(f"✅ Filtered data:")
print(f"   • Training windows after filtering: {len(train_dfs)}")
print(f"   • Test windows after filtering: {len(test_dfs)}")
print(f"   • Training classes after filtering: {dict(Counter(train_classes))}")
print(f"   • Test classes after filtering: {dict(Counter(test_classes))}")

balance_test = False  # Keep original test distribution
min_test_samples_per_class = 50  # Lower threshold for binary classification

print("Binary Classification Configuration:")
print("=" * 40)
print(f"All available classes: {all_available_classes}")
print(f"Selected classes for binary classification: {selected_classes}")
print(f"Strategy: One-vs-All binary classification")
print(f"Each classifier will distinguish one specific class from all others")

# Verify data availability
if not (train_dfs and test_dfs):
    print("❌ No data available. Run the data loading cell first.")
else:
    print(f"✅ Data ready: {len(train_dfs)} train, {len(test_dfs)} test windows")

    # Validate configuration
    config_valid = validate_configuration(selected_classes, test_classes, verbose=True)

    # Check fold information
    fold_available = test_fold_info is not None and len(test_fold_info) == len(test_dfs)
    if fold_available:
        unique_folds = sorted(set(test_fold_info))
        print(f"✅ Fold information: {len(unique_folds)} folds detected")

        # Print class distribution analysis
        print_class_distribution_analysis(
            test_classes=test_classes,
            test_fold_info=test_fold_info,
            selected_classes=selected_classes,
            verbose=True,
        )
    else:
        print("⚠️ No fold information available")

# Section 2: Setup Binary Classification Framework

Configure the binary classification framework with one-vs-all strategy, define model architecture similar to the OTC notebook, and prepare data structures for training.

**Note**: The data from `load_3w_data()` is already windowed and processed - no additional windowing needed!

In [ ]:
# ============================================================
# DATA PREPARATION - Using Already Windowed Data
# ============================================================

import numpy as np
from sklearn.preprocessing import StandardScaler
from collections import Counter

print("Preparing Windowed Data for Binary Classification")
print("=" * 50)

# The data from load_3w_data() is already windowed and processed
# train_dfs and test_dfs are already prepared sequences, not raw DataFrames
print(f"✅ Data already windowed and processed:")
print(f"   • Training windows: {len(train_dfs)} sequences")
print(f"   • Test windows: {len(test_dfs)} sequences") 

# Import binary classifiers module
sys.path.append("../../../src")
from src.binary_classifiers import convert_windowed_dfs_to_arrays

# Convert training data
print("\n🔄 Converting training data...")
X_train_full, y_train_full = convert_windowed_dfs_to_arrays(train_dfs, train_classes)

# Convert test data  
print("\n🔄 Converting test data...")
X_test_full, y_test_full = convert_windowed_dfs_to_arrays(test_dfs, test_classes)

# Display results
print(f"\n📊 Final Data Summary:")
print(f"   • Training data: {X_train_full.shape}")
print(f"   • Test data: {X_test_full.shape}")
print(f"   • Training classes: {dict(Counter(y_train_full))}")
print(f"   • Test classes: {dict(Counter(y_test_full))}")

# Store fold information for sequences (1:1 mapping since data is already processed)
train_fold_info_sequences = train_fold_info if train_fold_info else [0] * len(X_train_full)
test_fold_info_sequences = test_fold_info if test_fold_info else [0] * len(X_test_full)

print(f"   • Fold mapping: {len(train_fold_info_sequences)} train, {len(test_fold_info_sequences)} test")

In [ ]:
# ============================================================
# BINARY CLASSIFICATION MODEL ARCHITECTURE - PyTorch
# ============================================================

import torch
import torch.nn as nn

# Import binary classification framework
from src.binary_classifiers import (
    BinaryLSTMClassifier,
    create_binary_lstm_classifier,
    prepare_binary_data,
    train_pytorch_model
)

print("Binary Classification Framework Setup Complete")
print("=" * 50)
print("✅ Model architecture imported")
print("✅ Data preparation functions ready")
print("✅ Training functions available")
print("✅ Ready for training binary classifiers")

# Section 3: Train Binary Classifiers (One vs All)

Train individual binary classifiers for each class using the LSTM-based architecture, implementing one-vs-all strategy with proper data balancing and cross-validation.

In [ ]:
# ============================================================
# TRAIN BINARY CLASSIFIERS - ONE VS ALL STRATEGY (PyTorch)
# ============================================================

import torch
from src.binary_classifiers import train_binary_classifiers

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Training parameters
epochs = 50
batch_size = 32
learning_rate = 0.001
patience = 10

# Check if we have enough data
if len(X_train_full) == 0 or len(X_test_full) == 0:
    print("❌ No data available for training. Please check data preparation.")
    print("Debug Information:")
    print(f"   • X_train_full shape: {X_train_full.shape}")
    print(f"   • X_test_full shape: {X_test_full.shape}")
    print(f"   • Original train_dfs length: {len(train_dfs) if train_dfs else 0}")
    print(f"   • Original test_dfs length: {len(test_dfs) if test_dfs else 0}")
    
    print("\nPossible solutions:")
    print("   1. Run the data loading cell (cell 3) to load the windowed data")
    print("   2. Check if load_3w_data() is working correctly")
    print("   3. Verify the supervised_classification module is available")
    print("   4. Check the data conversion process in cell 7")
else:
    print(f"Training data shape: {X_train_full.shape}")
    print(f"Test data shape: {X_test_full.shape}")
    print(f"Classes to train: {selected_classes}")
    print()

    # Train all binary classifiers using the high-level function
    # Test set will be used as validation for early stopping
    binary_classifiers, training_results, training_histories = train_binary_classifiers(
        X_train_full, y_train_full, X_test_full, y_test_full, selected_classes,
        device,  # Pass device as positional argument
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        patience=patience
    )

    if training_results:
        print("\nTraining Performance Summary:")
        for class_num in sorted(training_results.keys()):
            results = training_results[class_num]
            print(f"   Class {class_num}: Train Acc={results['train_accuracy']:.3f}, "
                  f"Train F1={results['train_f1']:.3f}, "
                  f"Val Acc={results['val_accuracy']:.3f}, "
                  f"Val F1={results['val_f1']:.3f}, "
                  f"Samples={results['train_samples']}")

In [ ]:
# ============================================================
# EVALUATE BINARY CLASSIFIERS (PyTorch)
# ============================================================

from src.binary_classifiers import evaluate_binary_classifiers

# Evaluate all binary classifiers using the high-level function
evaluation_results = evaluate_binary_classifiers(
    binary_classifiers, X_test_full, y_test_full, device=device
)

# Section 4: Evaluate Binary Classifiers Performance

Evaluate each binary classifier's performance using accuracy, precision, recall, and F1-score metrics across different cross-validation folds.

In [ ]:
# ============================================================
# EVALUATE BINARY CLASSIFIERS ON TEST DATA
# ============================================================

# Storage for test results
test_results = {}
fold_results = {}

print("🎯 EVALUATING BINARY CLASSIFIERS ON TEST DATA")
print("=" * 50)

if not binary_classifiers:
    print("❌ No trained classifiers available. Please run training first.")
else:
    print(f"Evaluating {len(binary_classifiers)} trained classifiers")
    print(f"Test data shape: {X_test_full.shape}")
    print(f"Available folds: {sorted(set(test_fold_info_sequences))}")
    print()

    # Evaluate each trained classifier
    for target_class in sorted(binary_classifiers.keys()):
        print(f"📊 Evaluating Classifier for Class {target_class}")
        print("-" * 35)
        
        model = binary_classifiers[target_class]
        
        # Prepare binary test data (no balancing for evaluation)
        X_test_binary, y_test_binary, test_indices = prepare_binary_data(
            X_test_full, y_test_full, target_class, balance_classes=False
        )
        
        print(f"Test data for class {target_class}:")
        print(f"   • Total test samples: {len(X_test_binary)}")
        print(f"   • Positive samples (class {target_class}): {np.sum(y_test_binary)}")
        print(f"   • Negative samples (others): {len(y_test_binary) - np.sum(y_test_binary)}")
        
        if len(X_test_binary) == 0:
            print(f"   ⚠️ No test data available for class {target_class}")
            continue
        
        try:
            # Get predictions
            test_pred = model.predict(X_test_binary, verbose=0)
            test_pred_binary = (test_pred > 0.5).astype(int).flatten()
            
            # Calculate overall metrics
            test_accuracy = accuracy_score(y_test_binary, test_pred_binary)
            test_precision = precision_score(y_test_binary, test_pred_binary, zero_division=0)
            test_recall = recall_score(y_test_binary, test_pred_binary, zero_division=0)
            test_f1 = f1_score(y_test_binary, test_pred_binary, zero_division=0)
            
            # Calculate confusion matrix
            cm = confusion_matrix(y_test_binary, test_pred_binary)
            
            # Store results
            test_results[target_class] = {
                'test_accuracy': test_accuracy,
                'test_precision': test_precision,
                'test_recall': test_recall,
                'test_f1': test_f1,
                'confusion_matrix': cm,
                'test_samples': len(X_test_binary),
                'positive_samples': np.sum(y_test_binary),
                'predictions': test_pred.flatten(),
                'true_labels': y_test_binary
            }
            
            print(f"   ✅ Overall Performance:")
            print(f"      • Accuracy: {test_accuracy:.3f}")
            print(f"      • Precision: {test_precision:.3f}")
            print(f"      • Recall: {test_recall:.3f}")
            print(f"      • F1-Score: {test_f1:.3f}")
            
            # Evaluate by fold if fold information is available
            if len(set(test_fold_info_sequences)) > 1:
                print(f"   📁 Performance by Fold:")
                fold_results[target_class] = {}
                
                for fold in sorted(set(test_fold_info_sequences)):
                    # Get indices for this fold
                    fold_mask = np.array(test_fold_info_sequences)[test_indices] == fold
                    
                    if np.sum(fold_mask) > 0:
                        fold_X = X_test_binary[fold_mask]
                        fold_y = y_test_binary[fold_mask]
                        
                        # Get predictions for this fold
                        fold_pred = model.predict(fold_X, verbose=0)
                        fold_pred_binary = (fold_pred > 0.5).astype(int).flatten()
                        
                        # Calculate fold metrics
                        fold_accuracy = accuracy_score(fold_y, fold_pred_binary)
                        fold_precision = precision_score(fold_y, fold_pred_binary, zero_division=0)
                        fold_recall = recall_score(fold_y, fold_pred_binary, zero_division=0)
                        fold_f1 = f1_score(fold_y, fold_pred_binary, zero_division=0)
                        
                        fold_results[target_class][fold] = {
                            'accuracy': fold_accuracy,
                            'precision': fold_precision,
                            'recall': fold_recall,
                            'f1': fold_f1,
                            'samples': len(fold_X),
                            'positive_samples': np.sum(fold_y)
                        }
                        
                        print(f"      Fold {fold}: Acc={fold_accuracy:.3f}, F1={fold_f1:.3f}, "
                              f"Samples={len(fold_X)} ({np.sum(fold_y)} positive)")
            
            print()
            
        except Exception as e:
            print(f"   ❌ Evaluation failed for class {target_class}: {str(e)}")
            continue

print("📈 EVALUATION SUMMARY")
print("=" * 25)

if test_results:
    print("Overall Test Performance:")
    print(f"{'Class':<6} {'Accuracy':<9} {'Precision':<10} {'Recall':<8} {'F1-Score':<9} {'Samples':<8}")
    print("-" * 60)
    
    avg_metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
    
    for class_num in sorted(test_results.keys()):
        results = test_results[class_num]
        print(f"{class_num:<6} {results['test_accuracy']:<9.3f} {results['test_precision']:<10.3f} "
              f"{results['test_recall']:<8.3f} {results['test_f1']:<9.3f} {results['test_samples']:<8}")
        
        avg_metrics['accuracy'].append(results['test_accuracy'])
        avg_metrics['precision'].append(results['test_precision'])
        avg_metrics['recall'].append(results['test_recall'])
        avg_metrics['f1'].append(results['test_f1'])
    
    print("-" * 60)
    print(f"{'Avg':<6} {np.mean(avg_metrics['accuracy']):<9.3f} {np.mean(avg_metrics['precision']):<10.3f} "
          f"{np.mean(avg_metrics['recall']):<8.3f} {np.mean(avg_metrics['f1']):<9.3f}")
    
else:
    print("No evaluation results available.")

# Section 5: Analyze Results and Visualizations

Create comprehensive visualizations comparing performance across classes and folds, analyze confusion matrices, and provide performance summary tables.

In [ ]:
# ============================================================
# VISUALIZATION AND ANALYSIS
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

def plot_confusion_matrices():
    """Plot confusion matrices for all binary classifiers"""
    if not evaluation_results:
        print("No evaluation results available for visualization")
        return
    
    n_classifiers = len(evaluation_results)
    cols = min(3, n_classifiers)
    rows = (n_classifiers + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
    if n_classifiers == 1:
        axes = [axes]
    elif rows == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()
    
    for idx, (class_num, results) in enumerate(sorted(evaluation_results.items())):
        ax = axes[idx]
        
        # Calculate confusion matrix from predictions
        y_true = results['true_labels']
        y_pred = results['predictions']
        cm = confusion_matrix(y_true, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                   xticklabels=['Other', f'Class {class_num}'],
                   yticklabels=['Other', f'Class {class_num}'])
        
        ax.set_title(f'Class {class_num} vs All\nAcc: {results["accuracy"]:.3f}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
    
    # Hide unused subplots
    for idx in range(n_classifiers, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.suptitle('Confusion Matrices - Binary Classifiers', y=1.02, fontsize=16)
    plt.show()

def plot_performance_comparison():
    """Plot performance metrics comparison across classes"""
    if not evaluation_results:
        print("No evaluation results available for visualization")
        return
    
    # Prepare data for plotting
    classes = sorted(evaluation_results.keys())
    metrics = ['accuracy', 'precision', 'recall', 'f1_score']
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    
    data = []
    for class_num in classes:
        for metric, name in zip(metrics, metric_names):
            data.append({
                'Class': f'Class {class_num}',
                'Metric': name,
                'Value': evaluation_results[class_num][metric]
            })
    
    df_metrics = pd.DataFrame(data)
    
    # Create grouped bar plot
    plt.figure(figsize=(12, 6))
    sns.barplot(data=df_metrics, x='Class', y='Value', hue='Metric')
    plt.title('Binary Classifier Performance Comparison')
    plt.ylabel('Score')
    plt.xlabel('Fault Class')
    plt.legend(title='Metrics', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

def plot_fold_performance():
    """Plot performance across folds if available"""
    if not fold_results:
        print("No fold results available for visualization")
        return
    
    # Prepare fold data
    fold_data = []
    for class_num, class_folds in fold_results.items():
        for fold, metrics in class_folds.items():
            fold_data.append({
                'Class': f'Class {class_num}',
                'Fold': f'Fold {fold}',
                'Accuracy': metrics['accuracy'],
                'F1-Score': metrics['f1'],
                'Samples': metrics['samples']
            })
    
    if not fold_data:
        print("No fold data to visualize")
        return
    
    df_folds = pd.DataFrame(fold_data)
    
    # Create subplot for accuracy and F1
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Accuracy by fold
    sns.boxplot(data=df_folds, x='Class', y='Accuracy', ax=ax1)
    ax1.set_title('Accuracy Distribution Across Folds')
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45)
    
    # F1-Score by fold
    sns.boxplot(data=df_folds, x='Class', y='F1-Score', ax=ax2)
    ax2.set_title('F1-Score Distribution Across Folds')
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()

# ============================================================
# GENERATE VISUALIZATIONS
# ============================================================

print("📊 GENERATING VISUALIZATIONS AND ANALYSIS")
print("=" * 45)

print("📊 GENERATING VISUALIZATIONS AND ANALYSIS")
print("=" * 45)

if evaluation_results:
    print("Creating confusion matrices...")
    plot_confusion_matrices()
    
    print("Creating performance comparison...")
    plot_performance_comparison()
    
    if fold_results:
        print("Creating fold performance analysis...")
        plot_fold_performance()
    else:
        print("No fold results available for fold analysis")
        
else:
    print("❌ No evaluation results available for visualization")
    print("Please run the evaluation section first")

In [ ]:
# ============================================================
# DETAILED ANALYSIS AND SUMMARY TABLES
# ============================================================

def create_detailed_summary():
    """Create detailed summary tables and analysis"""
    print("📋 DETAILED PERFORMANCE ANALYSIS")
    print("=" * 40)
    
    if not test_results:
        print("No results available for analysis")
        return
    
    # Combined training and test results
    print("\n🔍 COMBINED TRAINING & VALIDATION PERFORMANCE:")
    print(f"{'Class':<6} {'Train Acc':<10} {'Val Acc':<9} {'Train F1':<9} {'Val F1':<8} {'Train Samples':<8}")
    print("-" * 65)
    
    for class_num in sorted(test_results.keys()):
        test_res = test_results[class_num]
        train_res = training_results.get(class_num, {})
        
        train_acc = train_res.get('train_accuracy', 0.0)
        train_f1 = train_res.get('train_f1', 0.0)
        val_acc = train_res.get('val_accuracy', 0.0)
        val_f1 = train_res.get('val_f1', 0.0)
        samples = train_res.get('train_samples', 0)
        
        print(f"{class_num:<6} {train_acc:<10.3f} {val_acc:<9.3f} {train_f1:<9.3f} {val_f1:<8.3f} {samples:<8}")
    
    # Class difficulty analysis
    print("\n🎯 CLASS DIFFICULTY ANALYSIS:")
    print("(Based on F1-Score performance)")
    
    class_performance = [(class_num, results['test_f1']) for class_num, results in test_results.items()]
    class_performance.sort(key=lambda x: x[1], reverse=True)
    
    print(f"{'Rank':<5} {'Class':<6} {'F1-Score':<9} {'Assessment'}")
    print("-" * 35)
    
    for rank, (class_num, f1_score) in enumerate(class_performance, 1):
        if f1_score >= 0.8:
            assessment = "Excellent"
        elif f1_score >= 0.6:
            assessment = "Good"
        elif f1_score >= 0.4:
            assessment = "Fair"
        elif f1_score >= 0.2:
            assessment = "Poor"
        else:
            assessment = "Very Poor"
        
        print(f"{rank:<5} {class_num:<6} {f1_score:<9.3f} {assessment}")
    
    # Data balance analysis
    print("\n⚖️ DATA BALANCE ANALYSIS:")
    print(f"{'Class':<6} {'Test +ve':<9} {'Test Total':<11} {'Pos Rate':<9} {'Balance'}")
    print("-" * 50)
    
    for class_num in sorted(test_results.keys()):
        results = test_results[class_num]
        pos_samples = results['positive_samples']
        total_samples = results['test_samples']
        pos_rate = pos_samples / total_samples if total_samples > 0 else 0
        
        if pos_rate < 0.05:
            balance = "Very Imbalanced"
        elif pos_rate < 0.15:
            balance = "Imbalanced"
        elif pos_rate < 0.35:
            balance = "Moderate"
        else:
            balance = "Balanced"
        
        print(f"{class_num:<6} {pos_samples:<9} {total_samples:<11} {pos_rate:<9.3f} {balance}")
    
    # Fold consistency analysis (if available)
    if fold_results:
        print("\n📁 FOLD CONSISTENCY ANALYSIS:")
        print("(Standard deviation of F1-scores across folds)")
        print(f"{'Class':<6} {'Mean F1':<8} {'Std F1':<7} {'Consistency'}")
        print("-" * 35)
        
        for class_num in sorted(fold_results.keys()):
            fold_f1s = [metrics['f1'] for metrics in fold_results[class_num].values()]
            if fold_f1s:
                mean_f1 = np.mean(fold_f1s)
                std_f1 = np.std(fold_f1s)
                
                if std_f1 < 0.05:
                    consistency = "Very Stable"
                elif std_f1 < 0.1:
                    consistency = "Stable"
                elif std_f1 < 0.2:
                    consistency = "Moderate"
                else:
                    consistency = "Unstable"
                
                print(f"{class_num:<6} {mean_f1:<8.3f} {std_f1:<7.3f} {consistency}")

def create_recommendations():
    """Create recommendations based on results"""
    print("\n💡 RECOMMENDATIONS FOR IMPROVEMENT:")
    print("=" * 45)
    
    if not evaluation_results:
        return
    
    # Poor performing classes (based on validation F1)
    poor_classes = [class_num for class_num in evaluation_results.keys()
                   if training_results.get(class_num, {}).get('val_f1', 0.0) < 0.5]
    
    if poor_classes:
        print(f"🔴 Classes with poor validation performance (F1 < 0.5): {poor_classes}")
        print("   Recommendations:")
        print("   • Collect more training data for these fault types")
        print("   • Consider feature engineering specific to these faults")
        print("   • Try different model architectures or hyperparameters")
        print("   • Investigate data quality issues")
    
    # Imbalanced classes
    imbalanced_classes = [class_num for class_num, results in evaluation_results.items()
                         if results['positive_samples'] / results['test_samples'] < 0.1]
    
    if imbalanced_classes:
        print(f"\n⚖️ Highly imbalanced classes: {imbalanced_classes}")
        print("   Recommendations:")
        print("   • Apply advanced sampling techniques (SMOTE, ADASYN)")
        print("   • Use cost-sensitive learning")
        print("   • Consider ensemble methods")
        print("   • Adjust classification thresholds")
    
    # General recommendations based on validation performance
    val_f1_scores = [training_results.get(class_num, {}).get('val_f1', 0.0) 
                     for class_num in evaluation_results.keys()]
    avg_f1 = np.mean(val_f1_scores) if val_f1_scores else 0.0
    print(f"\n📊 Overall average validation F1-score: {avg_f1:.3f}")
    
    if avg_f1 < 0.6:
        print("🔶 Overall performance needs improvement:")
        print("   • Consider increasing model complexity")
        print("   • Try different sequence lengths")
        print("   • Experiment with different feature combinations")
        print("   • Apply data augmentation techniques")
    elif avg_f1 < 0.8:
        print("🟡 Performance is moderate - room for improvement:")
        print("   • Fine-tune hyperparameters")
        print("   • Try ensemble methods")
        print("   • Consider transfer learning approaches")
    else:
        print("🟢 Good overall performance!")
        print("   • Consider deploying the model")
        print("   • Monitor performance on new data")

# ============================================================
# GENERATE DETAILED ANALYSIS
# ============================================================

create_detailed_summary()
create_recommendations()

print("\n" + "=" * 60)
print("✅ BINARY CLASSIFICATION ANALYSIS COMPLETE")
print("=" * 60)
print(f"Successfully trained and evaluated {len(binary_classifiers)} binary classifiers")
print("Check the visualizations and analysis above for detailed insights")

# Summary and Conclusions

This notebook successfully implemented a binary classification framework using a **one-vs-all strategy** for fault detection in the 3W dataset using **PyTorch**. 

## Key Achievements

1. **Data Integration**: Successfully combined the structured data loading approach from the supervised learning notebook with the LSTM architecture from the OTC notebook

2. **Binary Classification Framework**: Implemented individual binary classifiers for each fault class, allowing for:
   - Independent optimization per fault type
   - Clear interpretation of results per class
   - Detailed analysis of class-specific challenges

3. **PyTorch Implementation**: 
   - Custom BinaryLSTMClassifier with LSTM layers, batch normalization, and dropout
   - Proper training loop with validation and early stopping
   - GPU acceleration support for faster training
   - Comprehensive model evaluation and metrics tracking

## Architecture Benefits

- **LSTM-based models** effectively capture temporal patterns in time series data
- **Proper train/test separation** ensures realistic performance estimates
- **Cross-validation structure** provides robust evaluation across different data folds
- **Windowed sequences** enable effective pattern recognition in fault detection
- **PyTorch flexibility** allows for custom architectures and advanced training techniques

## Next Steps

- Implement ensemble methods combining multiple binary classifiers
- Experiment with different sequence lengths and feature combinations
- Apply advanced sampling techniques for imbalanced classes
- Consider multi-label classification approaches for cases with multiple simultaneous faults
- Explore attention mechanisms and transformer architectures

This framework provides a solid foundation for fault detection in oil well monitoring systems and can be extended to other time series classification tasks.